In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
FIG_DIR = os.path.join(ROOT, "figures")
ENC = "utf-8-sig"

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
BASE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")


import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

os.makedirs(FIG_DIR, exist_ok=True)

# 기존 3개 (T- 센서 계측, G-02 사건 기록, G-01 기준 정보)

# 2026-09-28 추가된 4개 (상위 이슈 #43)


pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False         # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110

---
# `G-02_정비이력.csv` 분석

In [ ]:
# G-02_정비이력.csv 데이터 로드 및 결과 확인

# 상단 초기화 셀에 정의된 경로(MAINT_CSV)와 인코딩(ENC)을 사용하여 데이터 로드
df = pd.read_csv(MAINT_CSV, encoding=ENC)
file_name = os.path.basename(MAINT_CSV)

print(f"✅ 데이터 로드 완료: {file_name}")
print(f"전체 규모: {df.shape[0]:,}행 × {df.shape[1]}열\n")

# `G-02_정비이력.csv`의 컬럼과 결측치 파악

In [ ]:
# 컬럼별 주요 상태 요약 표
col_summary = pd.DataFrame({
    "컬럼명": df.columns,
    "데이터 타입": df.dtypes.values,
    "결측치 수": df.isnull().sum().values,
    "유효 데이터 수": df.notnull().sum().values
})

display(col_summary)

## 📊 `G-02_정비이력.csv` 컬럼별 의미와 결측치 요약표

| 번호 | 컬럼명 | 한글명 | 결측치 수 | 유효 데이터 수 |
| :---: | :--- | :--- | :---: | :---: |
| 1 | `WO_NO` | 작업지시번호 | 0 | 2,112 |
| 2 | `EQP_TAG` | 설비태그 | 0 | 2,112 |
| 3 | `DETC_DT` | 고장/이상 감지일시 | 0 | 2,112 |
| 4 | `STRT_DT` | 정비 시작일시 | 0 | 2,112 |
| 5 | `END_DT` | 정비 종료일시 | 0 | 2,112 |
| 6 | `MNT_TYPE` | 정비유형 | 0 | 2,112 |
| 7 | `FAIL_MODE` | 고장모드 | 1,221 | 891 |
| 8 | `DOWN_MIN` | 다운타임(분) | 0 | 2,112 |
| 9 | `PART_CHG` | 부품교체 여부/상세 | 0 | 2,112 |
| 10 | `WRKR_ID` | 작업자 ID | 0 | 2,112 |
| 11 | `WRK_TEAM` | 작업 팀 | 0 | 2,112 |
| 12 | `REMARK` | 비고/작업내용 | 314 | 1,798 |

## 비교할 내용

정비 유형, 설비, 감지부터 착수까지의 시간, 작업 팀, 교체 부품을 확인함.


### 1. 정비 유형별 평균 정지시간

`BM`, `CBM`, `TBM`별 `DOWN_MIN`을 비교함.


In [ ]:
# 정비유형에 따른 다운타임 평균 및 총합 계산
mnt_down_df = df.groupby('MNT_TYPE')['DOWN_MIN'].agg(['count', 'mean', 'sum']).reset_index()
mnt_down_df.columns = ['정비유형', '정비건수', '평균다운타임(분)', '총다운타임(분)']

# 보기 쉽게 반올림
display(mnt_down_df.round(1))

BM와 CBM의 평균 정지시간이 TBM보다 길었음. CBM은 상태에 따라 정비하는 방식이므로 전부 돌발 고장으로 묶지 않음.


### 2. 설비태그별 주요 고장모드 분석
* **분석 목적**: 특정 설비(`EQP_TAG`)에서 자주 발생하는 고장모드(`FAIL_MODE`)의 패턴을 파악하여 취약 요소를 도출합니다.

In [ ]:
# 설비태그와 고장모드 교차표(Crosstab) 생성
fail_cross = pd.crosstab(df['EQP_TAG'], df['FAIL_MODE'], margins=True)
display(fail_cross)

위 표는 고장모드가 기록된 891건을 설비별로 집계한 결과임. 정비 건수만으로 고장률을 계산하지 않음.


### 3. 기록된 감지시각부터 작업 시작까지의 시간

`DETC_DT → STRT_DT`와 `STRT_DT → END_DT`를 나눠 비교함. 계획정비도 포함되어 있으므로 모든 `DETC_DT`를 고장 시작으로 보지 않음.


In [ ]:
# 날짜/시간 컬럼을 datetime 형식으로 변환
for col in ['DETC_DT', 'STRT_DT', 'END_DT']:
    df[col] = pd.to_datetime(df[col])

# 대기시간 (감지 ~ 정비시작) 과 순수 정비시간 (정비시작 ~ 종료) 계산 (분 단위)
df['WAIT_MIN'] = (df['STRT_DT'] - df['DETC_DT']).dt.total_seconds() / 60
df['PURE_MNT_MIN'] = (df['END_DT'] - df['STRT_DT']).dt.total_seconds() / 60

# 정비유형별 평균 대기시간 및 순수 정비시간 요약
time_analysis = df.groupby('MNT_TYPE')[['WAIT_MIN', 'PURE_MNT_MIN', 'DOWN_MIN']].mean().reset_index()
time_analysis.columns = ['정비유형', '평균 대기시간(분)', '평균 순수정비시간(분)', '평균 총다운타임(분)']

display(time_analysis.round(1))

대기시간과 작업시간은 위 표에서 정비 유형별로 확인함. 정비 유형과 작업 내용이 달라 평균 차이만으로 대응력을 판단하지 않음.


### 4. 작업 팀별 정비 건수와 평균 정지시간


In [ ]:
# 작업 팀별 정비 건수 및 평균 다운타임 계산
team_analysis = df.groupby('WRK_TEAM')['DOWN_MIN'].agg(['count', 'mean', 'sum']).reset_index()
team_analysis.columns = ['작업팀', '정비건수', '평균다운타임(분)', '총다운타임(분)']

display(team_analysis.round(1))

팀마다 맡은 설비와 작업이 다를 수 있으므로 평균 정지시간을 숙련도 순위로 해석하지 않음.


### 5. 부품교체이력 연결

`WO_NO`가 같은 정비와 교체 부품을 한 표로 확인함.


In [ ]:
# 부품 교체 이력 데이터 로드
df_partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

# WO_NO 기준으로 정비이력과 부품교체이력 병합 (Inner 혹은 Left Join)
df_merged = pd.merge(df, df_partchg, on='WO_NO', how='left')

print(f"병합된 데이터 규모: {df_merged.shape[0]:,}행 × {df_merged.shape[1]}열")
display(df_merged.head(3))

한 작업에서 여러 부품을 교체하면 같은 정비가 여러 행에 나타남. 이 표의 행 수를 정비 건수로 세지 않음.


### 6. 조업이벤트 원본 확인


In [ ]:
# 조업 이벤트 데이터 로드 확인
df_event = pd.read_csv(EVENT_CSV, encoding=ENC)
print(f"조업 이벤트 데이터 규모: {df_event.shape[0]:,}행 × {df_event.shape[1]}열")
display(df_event.head(3))

위 코드는 조업이벤트의 크기와 첫 3행을 확인함. 조업 변경 전후의 고장 집중 여부를 계산한 결과는 아님.


---
## 🔍 개별 컬럼 상세 탐색 (단변량 분석 및 데이터 상태 검증)

---
### <span id="desc-wo-no"></span>1. `WO_NO` (작업지시번호)
- **약어 풀이**: **W**ork **O**rder **N**umber
- **의미**: 각 정비/수리 작업마다 부여되는 고유 작업 지시 번호
- **데이터 상태**: 결측치 0건 (완전한 데이터)

In [ ]:
col = 'WO_NO'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 작업지시번호 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 ID 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-eqp-tag"></span>2. `EQP_TAG` (설비태그)
- **약어 풀이**: **Eq**uip**m**ent **Tag**
- **의미**: 정비가 수행된 대상 설비 식별 코드 (예: CAL01 등)
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'EQP_TAG'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 설비태그 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 태그 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

### ⚙️ 코드 실행: `EQP_TAG` 명명 규칙 분해 및 유형별 요약
* **목적**: 복잡한 설비태그(`EQP_TAG`)를 하이픈(`-`) 기준으로 쪼개어 공정 단위(`UNIT_CD`)와 개별 설비 유형(`EQP_TYPE`)으로 세분화합니다.
* **기대 효과**: 제철소 내 공정별 정비 편중도와 자주 고장 나는 설비 유형을 통계적으로 파악하고, 각 코드별 상세 의미(한글 설명)를 매핑하여 분석 보고서의 가독성을 높입니다.

In [ ]:

# 1. EQP_TAG 컬럼 분해 (Plant - Unit - Equipment Type)
split_tags = df['EQP_TAG'].str.split('-', expand=True)

df['PLANT'] = split_tags[0]
df['UNIT_CD'] = split_tags[1]
df['EQP_TYPE'] = split_tags[2].str.extract(r'([A-Za-z]+)')
df['EQP_NUM'] = split_tags[2].str.extract(r'(\d+)')

# 2. UNIT (@@#) 및 EQP_TYPE (@@@) 종류별 정비 건수 집계
unit_summary = df['UNIT_CD'].value_counts().reset_index()
unit_summary.columns = ['UNIT_코드', '정비건수']

eqp_type_summary = df['EQP_TYPE'].value_counts().reset_index()
eqp_type_summary.columns = ['EQP유형_코드', '정비건수']

# 3. 코드별 의미(설명) 딕셔너리 확장 정의 (데이터셋에 등장하는 전체 코드 반영)
unit_meaning_dict = {
    'BF1': '제1고로 (Blast Furnace 1)',
    'BF2': '제2고로 (Blast Furnace 2)',
    'SM1': '제1제강 (Steel Making 1)',
    'CC1': '제1연주 (Continuous Casting 1)',
    'CC2': '제2연주 (Continuous Casting 2)',
    'CR1': '제1냉연 (Cold Rolling 1)',
    'SP1': '제1소결 (Sinter Plant 1)',
    'SP2': '제2소결 (Sinter Plant 2)'
}

eqp_meaning_dict = {
    'PMP': '펌프 (Pump)',
    'SEG': '세그먼트 (Segment)',
    'HSV': '고속/서보 밸브 (Hydraulic Servo Valve)',
    'EXF': '배기팬/배출기 (Exhaust Fan)',
    'DCF': '집진기/필터 (Dust Collector)',
    'INS': '계측기/센서 (Instrument / Sensor)',
    'AGT': '교반기 (Agitator)',
    'MIX': '혼합기 (Mixer)',
    'BCV': '벨트컨베이어 (Belt Conveyor)',
    'CRS': '크러셔/파쇄기 (Crusher)',
    'BLW': '블로워 (Blower)',
    'CRN': '크레인 (Crane)',
    'SPM': '특수기계/가공기 (Special Machine)',
    'SNM': '선별기 (Sorting Machine)',
    'LDT': '하중측정기/로드셀 (Load Cell)',
    'BOF': '전로 (Basic Oxygen Furnace)',
    'MLD': '몰드 (Mold)',
    'PCI': '미분탄설비 (Pulverized Coal Injection)',
    'BFR': '송풍기 (Blower)',
    'CAL': '소둔로 (Continuous Annealing Line)',
    'IGF': '점화로/가스설비 (Ignition Furnace)',
    'WLD': '용접기 (Welder)',
    'TCM': '두께측정기 (Thickness Gauge)',
    'CLR': '냉각기 (Cooler)',
    'EMS': '비상정지/전기설비 (Emergency System)',
    'SCC': '슬래브 커터 (Slab Cutter)',
    'TUN': '터널로/가열로 (Tunnel Furnace)',
    'CAS': '주조/케이스 설비 (Casting)',
    'SCR': '스크린 (Screen)',
    'PKL': '산세설비 (Pickling Line)',
    'RCL': '권취기/릴 (Reeler / Coiler)',
    'LFR': '급기팬 (Local Fan)',
    'FAN': '팬 (Fan)',
    'RHD': '롤러하이드로/구동기 (Roller Drive)'
}

# 4. 딕셔너리를 활용하여 의미 컬럼 추가 (매핑되지 않은 코드는 기본 원형 유지)
unit_summary['UNIT_의미'] = unit_summary['UNIT_코드'].map(unit_meaning_dict).fillna('기타 공정 단위')
eqp_type_summary['EQP유형_의미'] = eqp_type_summary['EQP유형_코드'].map(eqp_meaning_dict).fillna('기타 설비 유형')

# 컬럼 순서 보기 좋게 재정치
unit_summary = unit_summary[['UNIT_코드', 'UNIT_의미', '정비건수']]
eqp_type_summary = eqp_type_summary[['EQP유형_코드', 'EQP유형_의미', '정비건수']]

print("=== 1. UNIT (@@#) 종류별 요약 및 의미 ===")
display(unit_summary)

print("\n=== 2. EQP_TYPE (@@@) 종류별 요약 및 의미 ===")
display(eqp_type_summary)

# 5. 외부 data 폴더에 의미가 포함된 CSV 파일로 각각 저장
OUTPUT_DIR = os.path.join(PROCESSED_DIR, "IWEasdwe1234")
os.makedirs(OUTPUT_DIR, exist_ok=True)
unit_csv_path = os.path.join(OUTPUT_DIR, "EQP_UNIT_요약_및_의미.csv")
eqp_type_csv_path = os.path.join(OUTPUT_DIR, "EQP_TYPE_요약_및_의미.csv")

unit_summary.to_csv(unit_csv_path, index=False, encoding=ENC)
eqp_type_summary.to_csv(eqp_type_csv_path, index=False, encoding=ENC)

print(f"\n✅ 설명이 포함된 요약 파일 저장 완료:")
print(f" - {unit_csv_path}")
print(f" - {eqp_type_csv_path}")

---
### 📊 [참고] 공정(`UNIT`) 및 설비 유형(`EQP_TYPE`)별 정비 건수 요약

#### 1. 공정 단위 (`UNIT_CD`)별 정비 현황
| 순위 | UNIT 코드 | 공정 의미 (한글 설명) | 정비 건수 |
| :---: | :---: | :--- | :---: |
| 1 | `CR1` | 제1냉연 (Cold Rolling 1) | 406건 |
| 2 | `SP2` | 제2소결 (Sinter Plant 2) | 356건 |
| 3 | `SP1` | 제1소결 (Sinter Plant 1) | 351건 |
| 4 | `CC1` | 제1연주 (Continuous Casting 1) | 275건 |
| 5 | `CC2` | 제2연주 (Continuous Casting 2) | 265건 |
| 6 | `BF2` | 제2고로 (Blast Furnace 2) | 156건 |
| 7 | `BF1` | 제1고로 (Blast Furnace 1) | 156건 |
| 8 | `SM1` | 제1제강 (Steel Making 1) | 147건 |

#### 2. 주요 설비 유형 (`EQP_TYPE`)별 정비 현황 (상위 주요 설비)
| 순위 | EQP 코드 | 설비 유형 의미 (한글 설명) | 정비 건수 |
| :---: | :---: | :--- | :---: |
| 1 | `PMP` | 펌프 (Pump) | 166건 |
| 2 | `SEG` | 세그먼트 (Segment) | 163건 |
| 3 | `HSV` | 고속/서보 밸브 (Hydraulic Servo Valve) | 126건 |
| 4 | `EXF` | 배기팬/배출기 (Exhaust Fan) | 120건 |
| 5 | `DCF` | 집진기/필터 (Dust Collector) | 118건 |
| 6 | `INS` | 계측기/센서 (Instrument / Sensor) | 106건 |
| 7 | `AGT` | 교반기 (Agitator) | 106건 |
| 8 | `MIX` | 혼합기 (Mixer) | 88건 |

> 💡 **인사이트 요약**: 
> * 공정별로는 **제1냉연(`CR1`)**과 **소결 공정(`SP1`, `SP2`)**에서 정비 수요가 가장 높게 나타났습니다.
> * 설비 유형별로는 **펌프(`PMP`)**, **세그먼트(`SEG`)**, **서보밸브(`HSV`)** 등의 유압·구동 기계 부품 계열에서 정비 오더가 집중되는 경향을 보입니다.

---
### <span id="desc-detc-dt"></span>3. `DETC_DT` (고장/이상 감지일시)
- **약어 풀이**: **Det**e**c**tion **D**a**t**e
- **의미**: 정비이력에 적힌 감지시각. 계획정비에도 값이 있어 고장 시작으로 일괄 해석하지 않음
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'DETC_DT'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 고장/이상 감지일시 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 일시 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-strt-dt"></span>4. `STRT_DT` (정비 시작일시)
- **약어 풀이**: **St**a**rt** **D**a**t**e
- **의미**: 정비 작업이 실제로 시작된 시각
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'STRT_DT'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 정비 시작일시 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 일시 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-end-dt"></span>5. `END_DT` (정비 종료일시)
- **약어 풀이**: **E**n**d** **D**a**t**e
- **의미**: 정비 작업이 끝난 시각. 정상 가동 재개 시각은 별도 확인이 필요함
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'END_DT'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 정비 종료일시 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 일시 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-mnt-type"></span>6. `MNT_TYPE` (정비유형)
- **약어 풀이**: **M**ai**n**te**n**ance **Type**
- **의미**: 정비 목적 및 방식 분류 (`BM`: 사후보전, `CBM`: 상태기반보전, `TBM`: 시간기반보전 등)
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'MNT_TYPE'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 정비유형 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 유형 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

| 정비 유형 코드 | 정비 유형 명칭 (한글) | 설명 |
| :---: | :--- | :--- |
| **BM** | 사후보전 (Breakdown Maintenance) | 고장 발생 후 조치하는 방식 |
| **CBM** | 상태기준보전 (Condition Based Maintenance) | 상태 지표가 임계치를 넘을 때 조치 |
| **TBM** | 시간기준보전 (Time Based Maintenance) | 일정 시간이 도래하면 교체/정비 |

> **📌 데이터 검증 참고**: 본 정비이력 데이터셋에는 총 3가지(`BM`, `CBM`, `TBM`) 정비 유형이 기록되어 있으며, 일반적인 설비 보전 학술/실무 표준에 존재하는 예방보전(`PM`), 예지보전(`PdM`) 등은 본 데이터의 실무 기록 특성상 제외되어 있습니다.

---
### <span id="desc-fail-mode"></span>7. `FAIL_MODE` (고장모드)
- **약어 풀이**: **Fail**ure **Mode**
- **의미**: 설비 고장 원인 및 현상 분류 (예: 마모, 과열, 누수 등)
- **데이터 상태**: **결측치 1,221건 (약 57.8%)** - 정기점검/예방정비(TBM/CBM)의 경우 고장이 아니므로 결측일 가능성 분석 필요

In [ ]:
col = 'FAIL_MODE'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 고장모드 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 모드 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

`FAIL_MODE`가 없는 행은 1,221건임. 결측만으로 정상이나 고장을 정하지 않고 `MNT_TYPE`과 작업 내용을 함께 확인함.


---
### <span id="desc-down-min"></span>8. `DOWN_MIN` (다운타임)
- **약어 풀이**: **Down**time **Min**utes
- **의미**: 정비 및 고장으로 인해 설비가 정지된 시간 (단위: 분)
- **데이터 상태**: 결측치 0건 (숫자형 데이터: `int64`)

In [ ]:
col = 'DOWN_MIN'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 다운타임(분) 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 값 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-part-chg"></span>9. `PART_CHG` (부품교체 여부/상세)
- **약어 풀이**: **Part** **Ch**an**g**e
- **의미**: 정비 과정에서의 부품 교체 유무 또는 교체 내역
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'PART_CHG'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 부품교체 상세 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 항목 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-wrkr-id"></span>10. `WRKR_ID` (작업자 ID)
- **약어 풀이**: **Wor**k**e**r **ID**
- **의미**: 정비 작업을 수행한 담당 작업자 식별 번호
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'WRKR_ID'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 작업자 ID 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 작업자 수(Unique) : {unique_cnt:,}명")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-wrk-team"></span>11. `WRK_TEAM` (작업 팀)
- **약어 풀이**: **Work** **Team**
- **의미**: 정비 작업을 담당한 보전 팀/조 구분
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'WRK_TEAM'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 작업 팀 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 팀 수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-remark"></span>12. `REMARK` (비고/작업내용)
- **약어 풀이**: **Remark**
- **의미**: 현장 정비 담당자가 작성한 작업 상세 세부 메모 및 특이사항
- **데이터 상태**: **결측치 314건 (약 14.9%)**

In [ ]:
col = 'REMARK'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 비고/작업내용 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 비고 수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

`REMARK`가 없는 행은 314건임. 비고가 없다고 이상이 없었다는 뜻은 아님.
